In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

raiz = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'analysis' / 'rtr_data.py').is_file()),
    None,
)
if raiz is None:
    raise RuntimeError('Abra o notebook na pasta do repositório.')
sys.path.insert(0, str(raiz / 'analysis'))
from rtr_data import load_dataset, read_manifest

exports = raiz / '.local' / 'exports'
if not (exports / 'latest.json').is_file():
    raise FileNotFoundError('Inicie o Observatório e aguarde a exportação de dados.')
manifesto = read_manifest(exports)
atributos = pd.DataFrame([
    {'arquivo': dataset['name'] + '.csv', 'posição': i, 'atributo': coluna['name']}
    for dataset in manifesto['datasets']
    for i, coluna in enumerate(dataset['columns'], start=1)
])
display(atributos)


# Métricas de vídeo e desempenho — últimos 30 dias

Fonte: snapshot exportado `active_video_metrics.csv`, Uma linha por vídeo. **Vídeos e canais excluídos são removidos antes da normalização e de todos os agrupamentos.**

**Normalização por canal:** `z = (valor − média do canal) / desvio padrão do canal`, usando desvio populacional (`ddof=0`) e apenas valores disponíveis de cada métrica. Z-score 0 é a média do canal; +1 é um desvio acima; −1 é um desvio abaixo. Valores ausentes e canais sem variação ficam sem z-score.

As médias conjuntas dão **peso igual a cada vídeo**, depois da normalização por canal. Os horários e dias referem-se à **publicação**, no fuso `America/Sao_Paulo`. Views são acumuladas até a coleta: vídeos mais antigos tiveram mais tempo de exposição. Estes agrupamentos descrevem a amostra e não isolam o efeito do horário, dia ou tag.

**Recorte de todas as análises: publicações de 17/10/2026 a 15/11/2026, inclusive, em Brasília.** O filtro por data de publicação é aplicado antes dos cálculos: médias e desvios usados nos z-scores são calculados exclusivamente com os vídeos desses 30 dias em cada canal. Views continuam sendo as métricas acumuladas disponíveis na coleta, e não o ganho de views durante o período.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

resultados = raiz / 'analysis' / 'resultados'
resultados.mkdir(exist_ok=True)

metricas_video = load_dataset('active_video_metrics', exports_dir=exports).copy()
assert metricas_video['deleted_at'].isna().all()
assert metricas_video['id'].is_unique, 'A base deve ter uma linha por vídeo.'
assert metricas_video['channel_id'].notna().all(), 'Vídeos sem canal.'
metricas_video['canal'] = metricas_video['channel_handle'].fillna(metricas_video['channel_id'])

# Deriva o calendário da data de publicação: segunda=0, domingo=6.
# Não utiliza weekday_brt, cuja convenção pode ser diferente.
metricas_video['publicado_brt'] = pd.to_datetime(
    metricas_video['published_at'], utc=True, errors='coerce', format='mixed'
).dt.tz_convert('America/Sao_Paulo')
# Período comum às análises; filtra ANTES de calcular os z-scores.
# Para reproduzir o estudo de caso, use '2026-11-15' com os dados daquela coleta.
data_escolhida = None  # ou 'AAAA-MM-DD'
ultima_publicacao = metricas_video['publicado_brt'].max()
if pd.isna(ultima_publicacao):
    raise ValueError('Colete vídeos com data de publicação antes de executar a análise.')
DATA_REFERENCIA = (
    ultima_publicacao.normalize()
    if data_escolhida is None
    else pd.Timestamp(data_escolhida, tz='America/Sao_Paulo').normalize()
)
inicio_30d = DATA_REFERENCIA - pd.Timedelta(days=29)
fim_exclusivo = DATA_REFERENCIA + pd.Timedelta(days=1)
cadastro_canais_ativos = metricas_video[['channel_id', 'canal']].drop_duplicates()
metricas_video = metricas_video.loc[
    metricas_video['publicado_brt'].ge(inicio_30d)
    & metricas_video['publicado_brt'].lt(fim_exclusivo)
].copy()
assert not metricas_video.empty, 'Nenhum vídeo disponível no período.'
assert metricas_video['publicado_brt'].ge(inicio_30d).all()
assert metricas_video['publicado_brt'].lt(fim_exclusivo).all()
print(f"Filtro: {inicio_30d:%d/%m/%Y} a {DATA_REFERENCIA:%d/%m/%Y}, inclusive (Brasília)")

dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']
faixas = [f'{h:02d}h–{h+2:02d}h' for h in range(0, 24, 2)]
metricas_video['dia_semana'] = pd.Categorical(
    metricas_video['publicado_brt'].dt.dayofweek.map(dict(enumerate(dias))),
    categories=dias, ordered=True
)
metricas_video['faixa_horaria'] = pd.cut(
    metricas_video['publicado_brt'].dt.hour,
    bins=range(0, 25, 2), labels=faixas, right=False
)

# Métricas de desempenho: preserva os valores originais e adiciona *_z.
metricas = [
    'views', 'likes', 'comment_count', 'shares', 'reposts', 'saves', 'downloads',
    'hashtag_count', 'like_view_ratio', 'video_likes_per_comment',
    'comment_view_ratio', 'share_view_ratio', 'comments_loaded'
]
metricas_video[metricas] = metricas_video[metricas].apply(pd.to_numeric, errors='coerce').astype('float64')
grupos = metricas_video.groupby('channel_id')[metricas]
medias_canal = grupos.transform('mean')
desvios_canal = grupos.transform('std', ddof=0)
z = (metricas_video[metricas] - medias_canal) / desvios_canal.replace(0, np.nan)
metricas_video[[f'{m}_z' for m in metricas]] = z
metricas_video['views_media_canal'] = medias_canal['views']
metricas_video['views_desvio_canal'] = desvios_canal['views']
metricas_video.to_csv(resultados / 'metricas_video_normalizadas.csv', index=False, encoding='utf-8-sig')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
def mostrar_tabela(df):
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(df.style.format(precision=3, na_rep='—'))

print(f"{len(metricas_video)} vídeos | {metricas_video['channel_id'].nunique()} canais")
print(f"Vídeos marcados como excluídos: {metricas_video['deleted_at'].notna().sum()}")
print(f"Sem views: {metricas_video['views'].isna().sum()} | Sem data: {metricas_video['publicado_brt'].isna().sum()}")
print('Métricas sem dados:', ', '.join(m for m in metricas if metricas_video[m].notna().sum() == 0))
print('Período de publicação:', metricas_video['publicado_brt'].min(), 'a', metricas_video['publicado_brt'].max())


## Tabela de métricas normalizadas por canal

`metricas_video` contém todos os campos originais e os z-scores `*_z`. A prévia abaixo mostra views; o CSV exportado contém todas as linhas e métricas. A tabela seguinte resume a população usada para normalizar cada canal.

In [ ]:
display(metricas_video[['id', 'canal', 'publicado_brt', 'views',
                       'views_media_canal', 'views_desvio_canal', 'views_z']].head(15))
resumo_canais = metricas_video.groupby('canal').agg(
    videos=('id', 'size'), videos_com_views=('views', 'count'),
    views_media=('views', 'mean'), views_mediana=('views', 'median'),
    views_desvio=('views', lambda s: s.std(ddof=0)),
    views_z_media=('views_z', 'mean')
)
mostrar_tabela(resumo_canais)


## 1. Média de views por dia da semana, para cada canal

Para cada canal e dia de publicação: **soma das views / quantidade de vídeos com views disponíveis**. Os rótulos `n=` mostram essa quantidade. Cada painel tem sua própria escala de views; dias sem vídeos ficam sem barra.

In [ ]:
canal_dia = metricas_video.groupby(['canal', 'dia_semana'], observed=False).agg(
    n_videos=('id', 'size'), n_com_views=('views', 'count'),
    views_total=('views', lambda s: s.sum(min_count=1)),
    views_media=('views', 'mean')
)
canal_dia.to_csv(resultados / 'views_por_canal_dia.csv', encoding='utf-8-sig')
canais = sorted(metricas_video['canal'].unique())
fig, axes = plt.subplots(int(np.ceil(len(canais) / 2)), 2,
                         figsize=(15, 3.6 * int(np.ceil(len(canais) / 2))), squeeze=False)
for ax, canal in zip(axes.flat, canais):
    tabela = canal_dia.loc[canal].reindex(dias)
    ax.bar(range(7), tabela['views_media'], color='#277DA1', width=.65)
    ax.set_xticks(range(7), [d[:3] for d in dias])
    ax.set_title('@' + canal, loc='left', fontweight='bold')
    ax.set_ylabel('Views médias / vídeo')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'.replace(',', '.')))
    ax.grid(axis='y', alpha=.2)
    ax.set_axisbelow(True)
    ax.margins(y=.22)
    for i, (valor, n) in enumerate(zip(tabela['views_media'], tabela['n_com_views'])):
        if pd.notna(valor):
            ax.annotate(f'n={int(n)}', (i, valor), xytext=(0, 5),
                        textcoords='offset points', ha='center', fontsize=8)
for ax in list(axes.flat)[len(canais):]:
    ax.set_visible(False)
fig.suptitle('Views médias por dia de publicação — cada canal | últimos 30 dias', fontsize=16, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, .97))
fig.savefig(resultados / '01_views_por_canal_dia.png', bbox_inches='tight')
plt.show()
mostrar_tabela(canal_dia)


## 2. Média de views normalizadas por dia da semana — todos os canais

Primeiro normaliza cada vídeo em relação ao seu canal; depois agrupa todos os vídeos por dia. A média é **soma dos z-scores / quantidade de vídeos com z-score disponível**. O eixo está em desvios padrão, não em quantidade de views.

In [ ]:
def resumir_z(coluna, ordem):
    return metricas_video.groupby(coluna, observed=False).agg(
        n_videos=('id', 'size'), n_com_z=('views_z', 'count'),
        n_canais=('channel_id', 'nunique'),
        views_z_soma=('views_z', lambda s: s.sum(min_count=1)),
        views_z_media=('views_z', 'mean'), views_z_mediana=('views_z', 'median')
    ).reindex(ordem)

def plotar_z(tabela, titulo, eixo_x, arquivo):
    fig, ax = plt.subplots(figsize=(12, 4.8))
    valores = tabela['views_z_media']
    cores = ['#277DA1' if pd.isna(v) or v >= 0 else '#D77A45' for v in valores]
    ax.bar(range(len(tabela)), valores, color=cores, width=.65)
    ax.axhline(0, color='#334155', linewidth=.8)
    ax.set_xticks(range(len(tabela)), tabela.index.astype(str))
    ax.set_xlabel(eixo_x)
    ax.set_ylabel('Média de views_z (desvios padrão)')
    ax.set_title(titulo + '\nÚltimos 30 dias: 17/10 a 15/11/2026', loc='left', fontweight='bold', pad=18)
    ax.grid(axis='y', alpha=.2)
    ax.set_axisbelow(True)
    ax.margins(y=.25)
    for i, (valor, n) in enumerate(zip(valores, tabela['n_com_z'])):
        if pd.notna(valor):
            ax.annotate(f'{valor:.2f}\nn={int(n)}', (i, valor),
                        xytext=(0, 5 if valor >= 0 else -5), textcoords='offset points',
                        ha='center', va='bottom' if valor >= 0 else 'top', fontsize=9)
    fig.tight_layout()
    fig.savefig(resultados / arquivo, bbox_inches='tight')
    plt.show()

dia_normalizado = resumir_z('dia_semana', dias)
dia_normalizado.to_csv(resultados / 'views_z_por_dia.csv', encoding='utf-8-sig')
plotar_z(dia_normalizado, 'Desempenho relativo por dia de publicação — todos os canais',
         'Dia da semana (Brasília)', '02_views_z_por_dia.png')
mostrar_tabela(dia_normalizado)


## 3. Média de views normalizadas por faixa de 2 horas — todos os canais

Faixas de publicação em Brasília: **00h–02h, 02h–04h, …, 22h–24h**. Cada faixa inclui o início e exclui o fim: 02h00 entra em 02h–04h. Usa a mesma normalização por canal do item 2.

In [ ]:
horario_normalizado = resumir_z('faixa_horaria', faixas)
horario_normalizado.to_csv(resultados / 'views_z_por_horario.csv', encoding='utf-8-sig')
plotar_z(horario_normalizado, 'Desempenho relativo por horário de publicação — todos os canais',
         'Faixa horária (Brasília)', '03_views_z_por_horario.png')
mostrar_tabela(horario_normalizado)


## 4. Média e mediana de views por tag — todos os canais

Usa as hashtags atuais de `tags_json` em `active_video_metrics.csv`. Cada vídeo conta uma vez por tag; um vídeo com várias tags participa de vários grupos. O z-score é o do vídeo, calculado **antes** de expandir as tags, para não dar mais peso a vídeos com muitas tags na normalização.

A tabela mostra todas as tags, ordenadas pela média de `views_z`, com média e mediana normalizadas e originais. `n_videos` e `n_canais` ajudam a identificar resultados baseados em poucas observações. Vídeos sem tags não entram nesta tabela.

In [ ]:
def ler_tags(valor):
    if pd.isna(valor):
        return []
    tags = json.loads(valor)
    if not isinstance(tags, list):
        raise ValueError('tags_json deve conter uma lista.')
    return sorted({t.strip().lstrip('#').casefold() for t in tags
                   if isinstance(t, str) and t.strip().lstrip('#')})

tags_por_video = metricas_video['tags_json'].map(ler_tags)
videos_tags = metricas_video[['id', 'channel_id', 'views', 'views_z']].assign(tag=tags_por_video).explode('tag')
videos_tags = videos_tags.dropna(subset=['tag']).drop_duplicates(['id', 'tag'])
resumo_tags = videos_tags.groupby('tag').agg(
    n_videos=('id', 'nunique'), n_canais=('channel_id', 'nunique'),
    n_com_views=('views', 'count'), n_com_z=('views_z', 'count'),
    views_z_media=('views_z', 'mean'), views_z_mediana=('views_z', 'median'),
    views_media=('views', 'mean'), views_mediana=('views', 'median')
).sort_values(['views_z_media', 'n_videos'], ascending=[False, False])
resumo_tags.to_csv(resultados / 'views_por_tag.csv', encoding='utf-8-sig')
print(f"{len(resumo_tags)} tags | {len(videos_tags)} vínculos vídeo–tag | {(tags_por_video.str.len() == 0).sum()} vídeos sem tags")
mostrar_tabela(resumo_tags)


## Verificação dos cálculos

Confere a normalização por canal, os denominadores das médias, as faixas horárias e a ausência de duplicações nos vínculos vídeo–tag. CSVs e gráficos são salvos em `analysis/resultados` ao executar o notebook.

In [ ]:
# Garante que nenhuma análise use vídeos fora do período ou excluídos.
assert metricas_video['publicado_brt'].ge(inicio_30d).all()
assert metricas_video['publicado_brt'].lt(fim_exclusivo).all()
assert metricas_video['deleted_at'].isna().all()
assert metricas_video['channel_deleted_at'].isna().all()
# Uma média zero e desvio 1 em cada canal/métrica com variância positiva.
for metrica in metricas:
    esperado = desvios_canal[metrica].gt(0) & metricas_video[metrica].notna()
    assert metricas_video.loc[~esperado, f'{metrica}_z'].isna().all()
    grupos_z = metricas_video.loc[esperado].groupby('channel_id')[f'{metrica}_z']
    assert np.allclose(grupos_z.mean(), 0, atol=1e-10)
    assert np.allclose(grupos_z.std(ddof=0), 1, atol=1e-10)
assert np.allclose(canal_dia['views_media'],
                   canal_dia['views_total'] / canal_dia['n_com_views'].replace(0, np.nan), equal_nan=True)
for tabela in (dia_normalizado, horario_normalizado):
    assert np.allclose(tabela['views_z_media'],
                       tabela['views_z_soma'] / tabela['n_com_z'].replace(0, np.nan), equal_nan=True)
assert dia_normalizado['n_videos'].sum() == metricas_video['publicado_brt'].notna().sum()
assert horario_normalizado['n_videos'].sum() == metricas_video['publicado_brt'].notna().sum()
assert not videos_tags.duplicated(['id', 'tag']).any()
assert len(videos_tags) == tags_por_video.str.len().sum()
assert pd.cut(pd.Series([0, 1, 2, 23]), range(0, 25, 2), labels=faixas, right=False).astype(str).tolist() == [faixas[0], faixas[0], faixas[1], faixas[-1]]
print('Verificações concluídas: normalização, médias, grupos horários e tags consistentes.')


## 5. Frequência de publicação por canal — últimos 30 dias

A janela abrange 30 dias em Brasília e termina na data de referência definida na carga.

Conta **dias com publicação**, não quantidade de vídeos: dez vídeos na mesma data contam como um dia. Para medir dias por semana sem misturar semanas parciais, calcula as **24 janelas móveis de 7 dias** inteiramente contidas nos 30 dias (17–23/10, 18–24/10, …, 09–15/11). Cada janela contém uma ocorrência de cada dia da semana.

A tabela resume média, mediana e desvio padrão populacional (`ddof=0`) dessas 24 contagens, em **dias por semana**, e ordena pela média decrescente. Janelas se sobrepõem; o desvio descreve a variação das contagens, não a incerteza da média. Também mostra os dias da semana observados e o total de datas com publicação no período.

Dias sem vídeos na base contam como zero. Os resultados descrevem apenas os vídeos coletados; coleta incompleta pode subestimar a frequência. O dia 15/11 inclui somente os dados disponíveis na exportação.

In [ ]:
# Frequência bruta: não usa nenhuma coluna de z-score.
# Reutiliza o mesmo período definido na carga da base.
videos_30d = metricas_video.loc[
    metricas_video['publicado_brt'].ge(inicio_30d)
    & metricas_video['publicado_brt'].lt(fim_exclusivo)
].copy()
assert videos_30d['deleted_at'].isna().all()
assert videos_30d['channel_deleted_at'].isna().all()
videos_30d['data_publicacao'] = videos_30d['publicado_brt'].dt.normalize()

# Todos os canais da base, inclusive os que não publicaram no período.
calendario_30d = pd.date_range(inicio_30d, DATA_REFERENCIA, freq='D')
cadastro_canais = cadastro_canais_ativos.set_index('channel_id')
videos_por_data = pd.crosstab(videos_30d['data_publicacao'], videos_30d['channel_id']).reindex(
    index=calendario_30d, columns=cadastro_canais.index, fill_value=0
)
dias_com_post = videos_por_data.gt(0).astype(int)
# Soma de indicadores diários: cada dia vale no máximo 1, mesmo com muitos vídeos.
dias_por_janela = dias_com_post.rolling(window=7, min_periods=7).sum().iloc[6:].astype(int)
dias_por_janela.index.name = 'fim_janela_brt'

frequencia_canais = cadastro_canais.copy()
frequencia_canais['media_dias_por_semana'] = dias_por_janela.mean()
frequencia_canais['mediana_dias_por_semana'] = dias_por_janela.median()
frequencia_canais['desvio_padrao_dias_por_semana'] = dias_por_janela.std(ddof=0)
frequencia_canais['min_dias_por_semana'] = dias_por_janela.min()
frequencia_canais['max_dias_por_semana'] = dias_por_janela.max()
frequencia_canais['n_janelas_7d'] = len(dias_por_janela)
frequencia_canais['dias_com_post_30d'] = dias_com_post.sum()
frequencia_canais['videos_30d'] = videos_por_data.sum()
nomes_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']
semanas_observadas = videos_30d.groupby('channel_id')['publicado_brt'].agg(
    lambda s: sorted(set(s.dt.dayofweek))
)
frequencia_canais['n_dias_semana_observados'] = semanas_observadas.map(len).reindex(
    cadastro_canais.index, fill_value=0
)
frequencia_canais['dias_semana_observados'] = semanas_observadas.map(
    lambda ds: ', '.join(nomes_dias[d] for d in ds)
).reindex(cadastro_canais.index, fill_value='Nenhum')
frequencia_canais = frequencia_canais.sort_values(
    ['media_dias_por_semana', 'mediana_dias_por_semana', 'canal'],
    ascending=[False, False, True]
)

frequencia_canais.to_csv(resultados / 'frequencia_publicacao_30d.csv', encoding='utf-8-sig')
videos_por_data.rename(columns=cadastro_canais['canal']).to_csv(
    resultados / 'publicacoes_diarias_30d.csv', encoding='utf-8-sig'
)
dias_por_janela.rename(columns=cadastro_canais['canal']).to_csv(
    resultados / 'dias_publicacao_janelas_7d.csv', encoding='utf-8-sig'
)

# Confere cada janela por uma contagem independente das datas únicas nos vídeos.
for fim, linha in dias_por_janela.iterrows():
    amostra = videos_30d.loc[videos_30d['data_publicacao'].between(fim - pd.Timedelta(days=6), fim)]
    esperado = amostra.groupby('channel_id')['data_publicacao'].nunique().reindex(
        cadastro_canais.index, fill_value=0
    )
    assert np.array_equal(linha.to_numpy(), esperado.to_numpy())
assert len(dias_por_janela) == 24
assert dias_por_janela.ge(0).all().all() and dias_por_janela.le(7).all().all()
assert videos_por_data.to_numpy().sum() == len(videos_30d)
assert frequencia_canais['media_dias_por_semana'].is_monotonic_decreasing
print(f"Período: {inicio_30d:%d/%m/%Y} a {DATA_REFERENCIA:%d/%m/%Y} | {len(videos_30d)} vídeos | 24 janelas de 7 dias")
mostrar_tabela(frequencia_canais.reset_index(drop=True))
print('Verificação concluída: cada janela confere com as datas únicas de publicação.')


## 6. Explorações adicionais — últimos 30 dias

Todos os gráficos abaixo reutilizam a seleção de vídeos ativos da janela e os z-scores calculados dentro de cada canal nesse período. São associações descritivas: views são acumuladas e podem variar com a idade dos vídeos e com a cobertura da coleta.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import PercentFormatter

def salvar_grafico(fig, nome):
    fig.savefig(resultados / nome, dpi=150, bbox_inches='tight')
    plt.show()

periodo_graficos = f'{inicio_30d:%d/%m} a {DATA_REFERENCIA:%d/%m/%Y} · Brasília'
cores_canais = dict(zip(sorted(metricas_video['canal'].unique()), plt.get_cmap('tab10').colors))


### 6.1. Distribuição de views por canal

Mostra o vídeo típico e a dispersão, além da média. Caixas delimitam os quartis, a linha central é a mediana e os pontos são os vídeos. A escala logarítmica permite comparar canais de tamanhos diferentes; os valores continuam sendo views originais. Canais ordenados pela mediana.

In [ ]:
ordem_views = metricas_video.groupby('canal')['views'].median().sort_values().index
fig, ax = plt.subplots(figsize=(12, 6))
amostras = [metricas_video.loc[metricas_video['canal'].eq(c), 'views'].dropna() for c in ordem_views]
assert all(s.gt(0).all() for s in amostras), 'A escala log exige views positivas.'
caixas = ax.boxplot(amostras, vert=False, patch_artist=True, showfliers=False,
                    medianprops={'color': '#172554', 'linewidth': 2})
rng = np.random.default_rng(42)
for i, (canal, valores, caixa) in enumerate(zip(ordem_views, amostras, caixas['boxes']), start=1):
    caixa.set_facecolor(cores_canais[canal])
    caixa.set_alpha(.35)
    ax.scatter(valores, i + rng.uniform(-.13, .13, len(valores)), s=15,
               color=cores_canais[canal], alpha=.65)
ax.set_yticks(range(1, len(ordem_views)+1),
              [f'@{c} (n={len(s)})' for c,s in zip(ordem_views, amostras)])
ax.set_xscale('log')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.0f}'.replace(',', '.')))
ax.set_xlabel('Views por vídeo · escala logarítmica')
ax.set_title('Vídeo típico e dispersão das views por canal\n' + periodo_graficos, loc='left', fontweight='bold')
ax.grid(axis='x', alpha=.2)
fig.tight_layout()
salvar_grafico(fig, '04_distribuicao_views_canais.png')


### 6.2. Dia × horário: onde o desempenho relativo foi maior?

Cada célula mostra a média do z-score de views e o número de vídeos. Apenas combinações com **3 ou mais vídeos** são coloridas; amostras menores ficam em cinza. Azul indica acima da média do canal; laranja, abaixo. Uma célula pode representar poucos canais: consulte a tabela exportada antes de generalizar.

In [ ]:
mapa_dia_hora = metricas_video.groupby(['dia_semana', 'faixa_horaria'], observed=False).agg(
    views_z_media=('views_z', 'mean'), n=('views_z', 'count'), n_canais=('channel_id', 'nunique')
)
mapa_dia_hora.to_csv(resultados / 'mapa_dia_horario.csv', encoding='utf-8-sig')
valores = mapa_dia_hora['views_z_media'].unstack().reindex(index=dias, columns=faixas)
contagens = mapa_dia_hora['n'].unstack().reindex(index=dias, columns=faixas)
visiveis = valores.where(contagens.ge(3))
limite = max(.1, np.nanmax(np.abs(visiveis.to_numpy())))
cmap = plt.get_cmap('RdBu').copy()
cmap.set_bad('#edf0f4')
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(np.ma.masked_invalid(visiveis.to_numpy()), cmap=cmap,
               norm=TwoSlopeNorm(vmin=-limite, vcenter=0, vmax=limite), aspect='auto')
for y in range(7):
    for x in range(12):
        n = int(contagens.iloc[y,x])
        v = visiveis.iloc[y,x]
        if pd.notna(v):
            ax.text(x, y, f'{v:+.2f}\nn={n}', ha='center', va='center', fontsize=9,
                    color='white' if abs(v) > .6*limite else '#172554')
        else:
            ax.text(x, y, f'n={n}', ha='center', va='center', color='#64748b', fontsize=8)
ax.set_xticks(range(12), faixas, rotation=35, ha='right')
ax.set_yticks(range(7), dias)
ax.set_title('Desempenho por dia e horário de publicação · mínimo 3 vídeos/célula\n' + periodo_graficos,
             loc='left', fontweight='bold', pad=15)
fig.colorbar(im, ax=ax, label='Média de views_z')
fig.tight_layout()
salvar_grafico(fig, '05_mapa_dia_horario.png')


### 6.3. Engajamento por mil views

Para cada vídeo calcula `métrica / views × 1.000`; depois toma a **mediana por canal**. Evita que o tamanho do canal domine a comparação. Cada coluna usa sua própria escala de cor. Valores ausentes não viram zero; apenas vídeos com views positivas entram no denominador.

In [ ]:
colunas_eng = {'likes': 'Curtidas', 'comment_count': 'Comentários', 'shares': 'Compartilhamentos', 'saves': 'Salvamentos'}
taxas_eng = metricas_video[['canal', *colunas_eng]].copy()
taxas_eng[list(colunas_eng)] = metricas_video[list(colunas_eng)].div(
    metricas_video['views'].where(metricas_video['views'].gt(0)), axis=0
) * 1000
engajamento_canais = taxas_eng.groupby('canal')[list(colunas_eng)].median().rename(columns=colunas_eng)
engajamento_canais = engajamento_canais.sort_values('Compartilhamentos', ascending=False)
engajamento_canais.to_csv(resultados / 'engajamento_por_mil_views.csv', encoding='utf-8-sig')
fig, axes = plt.subplots(1, 4, figsize=(14, 5.6), sharey=True)
for i,(nome, ax) in enumerate(zip(engajamento_canais.columns, axes)):
    vals = engajamento_canais[nome].to_numpy()
    ax.imshow(vals[:, None], cmap='Blues', aspect='auto', vmin=0, vmax=max(1, np.nanmax(vals)*1.15))
    for y,v in enumerate(vals):
        ax.text(0, y, f'{v:.1f}', ha='center', va='center',
                color='white' if v > .65*max(vals) else '#172554', fontsize=12)
    ax.set_xticks([])
    ax.set_title(nome)
    ax.set_yticks(range(len(engajamento_canais)), ['@'+c for c in engajamento_canais.index])
fig.suptitle('Mediana de interações por 1.000 views · escalas de cor por coluna\n' + periodo_graficos,
             fontsize=14, fontweight='bold')
fig.tight_layout(rect=(0,0,1,.91))
salvar_grafico(fig, '06_engajamento_por_mil_views.png')


### 6.4. Duração do vídeo × desempenho relativo

Compara views normalizadas entre faixas de duração. Pontos representam vídeos e as caixas mostram quartis e mediana; losangos indicam a média. As amostras podem misturar canais e temas diferentes; o gráfico não mede o efeito causal da duração.

In [ ]:
rotulos_duracao = ['<60 s', '60–89 s', '90–119 s', '120–149 s', '150+ s']
base_duracao = metricas_video.assign(faixa_duracao=pd.cut(
    metricas_video['duration_seconds'].where(metricas_video['duration_seconds'].ge(0)),
    [0,60,90,120,150,np.inf], labels=rotulos_duracao, right=False
))
resumo_duracao = base_duracao.groupby('faixa_duracao', observed=False).agg(
    n=('views_z', 'count'), n_canais=('channel_id', 'nunique'),
    views_z_media=('views_z', 'mean'), views_z_mediana=('views_z', 'median')
)
resumo_duracao.to_csv(resultados / 'desempenho_por_duracao.csv', encoding='utf-8-sig')
amostras_duracao = [base_duracao.loc[base_duracao['faixa_duracao'].eq(f), 'views_z'].dropna() for f in rotulos_duracao]
fig, ax = plt.subplots(figsize=(11,5.5))
ax.boxplot(amostras_duracao, patch_artist=True, showfliers=False, showmeans=True,
           boxprops={'facecolor':'#dcecf4'}, meanprops={'marker':'D','markerfacecolor':'#d77a45','markeredgecolor':'#d77a45'})
for i,vals in enumerate(amostras_duracao, start=1):
    ax.scatter(i+rng.uniform(-.17,.17,len(vals)), vals, s=16, alpha=.5, color='#277da1')
ax.axhline(0, color='#334155', linewidth=.8)
ax.set_xticks(range(1,6), [f'{f}\nn={len(v)}' for f,v in zip(rotulos_duracao,amostras_duracao)])
ax.set_ylabel('Views normalizadas por canal (z-score)')
ax.set_xlabel('Duração do vídeo · losango = média')
ax.set_title('Vídeos mais longos tiveram desempenho diferente?\n' + periodo_graficos, loc='left', fontweight='bold')
ax.grid(axis='y', alpha=.2)
fig.tight_layout()
salvar_grafico(fig, '07_duracao_desempenho.png')


### 6.5. Concentração: quanto das views depende de poucos vídeos?

Ordena os vídeos de cada canal do mais visto ao menos visto. O eixo horizontal é a parcela dos vídeos; o vertical é a parcela das views que eles acumulam. Curvas que sobem mais rapidamente indicam maior concentração. A legenda informa a parcela de views dos **10% de vídeos mais vistos**, arredondando a quantidade de vídeos para cima.

In [ ]:
concentracao = []
fig, ax = plt.subplots(figsize=(12,6))
for canal in sorted(cores_canais):
    vals = metricas_video.loc[metricas_video['canal'].eq(canal), 'views'].dropna().sort_values(ascending=False).to_numpy()
    if len(vals)==0 or vals.sum()<=0:
        continue
    k = int(np.ceil(len(vals)*.1))
    parcela_top = vals[:k].sum()/vals.sum()
    concentracao.append({'canal':canal, 'n_videos':len(vals), 'n_top_10pct':k, 'parcela_views_top_10pct':parcela_top})
    ax.plot(np.r_[0,np.arange(1,len(vals)+1)/len(vals)], np.r_[0,vals.cumsum()/vals.sum()],
            color=cores_canais[canal], linewidth=2,
            label=f'@{canal} · top {k}: {parcela_top:.0%}')
ax.plot([0,1],[0,1], '--', color='#94a3b8', label='Views igualmente distribuídas')
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlim(0,1)
ax.set_ylim(0,1.02)
ax.set_xlabel('Parcela dos vídeos, do mais visto ao menos visto')
ax.set_ylabel('Parcela acumulada das views')
ax.set_title('Concentração de views nos vídeos mais vistos\n' + periodo_graficos, loc='left', fontweight='bold')
ax.grid(alpha=.2)
ax.legend(loc='center left', bbox_to_anchor=(1.02,.5), fontsize=9)
fig.tight_layout()
salvar_grafico(fig, '08_concentracao_views.png')
concentracao_views = pd.DataFrame(concentracao).sort_values('parcela_views_top_10pct', ascending=False)
concentracao_views.to_csv(resultados / 'concentracao_views.csv', index=False, encoding='utf-8-sig')


### 6.6. Calendário de publicações

Cada célula representa um canal em uma data, com o número de vídeos publicados. Zero significa nenhuma publicação observada na base. Ordenado por quantidade de dias com publicação nos 30 dias; mostra pausas e concentração de postagens.

In [ ]:
calendario_posts = videos_por_data.rename(columns=cadastro_canais['canal']).T
ordem_calendario = calendario_posts.gt(0).sum(axis=1).sort_values(ascending=False).index
calendario_posts = calendario_posts.loc[ordem_calendario]
fig, ax = plt.subplots(figsize=(16,5.6))
im = ax.imshow(calendario_posts.to_numpy(), cmap='Blues', aspect='auto', vmin=0)
for y in range(len(calendario_posts)):
    for x in range(30):
        n = int(calendario_posts.iloc[y,x])
        if n:
            ax.text(x,y,str(n),ha='center',va='center',fontsize=9,
                    color='white' if n > .6*calendario_posts.to_numpy().max() else '#172554')
ax.set_yticks(range(len(calendario_posts)), ['@'+c for c in calendario_posts.index])
ax.set_xticks(range(30),[d.strftime('%d/%m') for d in calendario_posts.columns],rotation=60,ha='right')
ax.set_title('Ritmo de publicação: vídeos por dia\n' + periodo_graficos, loc='left',fontweight='bold',pad=15)
fig.colorbar(im,ax=ax,label='Vídeos publicados', shrink=.8)
fig.tight_layout()
salvar_grafico(fig, '09_calendario_publicacoes.png')


### 6.7. Tags com presença em vários canais

Exibe até 15 tags com maior média de views_z entre aquelas presentes em **pelo menos 10 vídeos e 2 canais**. Círculo é a média, losango é a mediana; o segmento entre eles é apenas a diferença entre as duas medidas, não um intervalo de confiança. O rótulo mostra quantidade de vídeos e canais. Tags que aparecem juntas não têm seus efeitos separados.

In [ ]:
tags_com_amostra = resumo_tags.loc[resumo_tags['n_com_z'].ge(10) & resumo_tags['n_canais'].ge(2)]
tags_grafico = tags_com_amostra.head(15).sort_values('views_z_media')
tags_com_amostra.to_csv(resultados / 'tags_amostra_minima.csv', encoding='utf-8-sig')
if not tags_grafico.empty:
    fig, ax = plt.subplots(figsize=(12,max(4, .4*len(tags_grafico)+1.8)))
    yy = np.arange(len(tags_grafico))
    ax.hlines(yy, tags_grafico['views_z_media'], tags_grafico['views_z_mediana'], color='#94a3b8', linewidth=2)
    ax.scatter(tags_grafico['views_z_media'], yy, color='#277da1', s=55, label='Média', zorder=3)
    ax.scatter(tags_grafico['views_z_mediana'], yy, color='#d77a45', marker='D', s=40, label='Mediana', zorder=3)
    ax.set_yticks(yy, [f'#{tag} · n={int(row.n_com_z)}, {int(row.n_canais)} canais' for tag,row in tags_grafico.iterrows()])
    ax.axvline(0,color='#334155',linewidth=.8)
    ax.set_xlabel('Views normalizadas por canal (z-score)')
    ax.set_title('Tags compartilhadas por canais · mínimo 10 vídeos e 2 canais\n' + periodo_graficos,loc='left',fontweight='bold')
    ax.grid(axis='x',alpha=.2)
    ax.legend(loc='lower right')
    fig.tight_layout()
    salvar_grafico(fig, '10_tags_amostra_minima.png')
else:
    print('Nenhuma tag atende aos critérios de amostra.')

# Verificações das agregações adicionais.
assert mapa_dia_hora['n'].sum() == metricas_video['views_z'].count()
assert resumo_duracao['n'].sum() == base_duracao.loc[base_duracao['faixa_duracao'].notna(),'views_z'].count()
assert calendario_posts.to_numpy().sum() == len(metricas_video)
assert concentracao_views['parcela_views_top_10pct'].between(0,1).all()
print('Gráficos adicionais gerados e agregações verificadas.')
